# S6E10 — 12 RealMLP-rich: a second training recipe

> **Pipeline position:** 09 `X_rich` → 10 RealMLP-rich → **12 RealMLP-rich, recipe v2** → 11 stack

| | |
|---|---|
| **Input** | `data/processed/X_rich_*.parquet` (from `09`), `y_train.parquet`; fold-0 checkpoints of `10` for the benchmark |
| **Output** | `preds/realmlp_rich_v2_folds/` and `preds/realmlp_rich_refgpu_folds/` (fold checkpoints); if accepted: `preds/oof_realmlp_rich_v2*.npy`, `test_realmlp_rich_v2*.npy` |
| **Runtime** | fold 0 ≈ a few minutes per recipe on an RTX 3070 Ti (4 networks); the timing probe prints the estimate |
| **Result** | see *Results & decisions* |

**Why:** `10` uses the community recipe unchanged (`lin_cos_log_15` learning-rate schedule, weight decay 0.0236, dropout schedule `expm4t`, 4 epochs). A second recipe seen in public work trains differently: **flat learning rate, then a cosine anneal** (`flat_anneal`), lighter weight decay (0.015), dropout that **switches off almost immediately** (`invsqrtp1e-3`) and **3 epochs**. If it ranks rows better, it improves the strongest network member of the stack without new features.

**What changes vs `10`:** only five parameters (table in section 2). Features, `nn_frame`, in-fold target encoding, folds and the number of networks (**4**, as in the benchmark) are identical, so a difference is attributable to the recipe.

**Fair benchmark (why the reference recipe is re-run here):** the benchmark seeds of `10` were trained on the Mac CPU; this notebook runs on a GPU. GPU and CPU arithmetic differ slightly, so the reference recipe is trained again **on the same machine with the same seed** and the new recipe is compared with it (a paired comparison). The Mac numbers stay in the table as a check that the GPU reproduces them.

**Acceptance (project rule):** fold 0 first; a gain below +0.0002 on one fold is noise. Only if fold 0 passes: 8 networks, then all 5 folds and the stack (`11`).

> **Glossary:** *Recipe* = training settings (schedules, regularisation, epochs), as opposed to the input features. *Schedule* = how a value (learning rate, dropout, weight decay) changes over training time `t` from 0 to 1. *Paired comparison* = both variants trained under identical conditions (data, fold, seed, machine), so the difference is not machine noise.

In [1]:
import torch          # first import: on the Mac the kernel crashes if torch comes after numpy/sklearn (OpenMP conflict)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, pandas as pd, time, gc, warnings, inspect
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
import importlib.metadata
PYTABKIT_VERSION = importlib.metadata.version('pytabkit')
from pytabkit import RealMLP_TD_Classifier
import pytabkit.models.training.scheduling as pt_sched
warnings.filterwarnings('ignore')

SMOKE = False        # True: 60k training rows, 1 epoch, 2 networks: only checks that the code runs
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'      # 'mps' is not used (unstable in this environment)
N_ENS = 4            # fixed, NOT 8 on GPU as in 10: the recipe effect must not mix with the ensemble-size effect
SEED = 0             # one notebook run = one seed (as in 10); fold-0 benchmark uses seed 0 for both recipes
RUN_REF = True       # also train the reference recipe of 10 on fold 0 on this machine (paired comparison)
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '', '| torch', torch.__version__,
      '| pytabkit', PYTABKIT_VERSION, '| networks per model:', N_ENS)

N_FOLDS, RANDOM_STATE = 5, 42
CANDS = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDS if (p / 'X_rich_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds')] if p.exists()), Path('preds')); PREDS_DIR.mkdir(exist_ok=True)
X = pd.read_parquet(PROC_DIR / 'X_rich_train.parquet'); X_test = pd.read_parquet(PROC_DIR / 'X_rich_test.parquet')
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
# The project-wide split: identical folds in every notebook keep OOF predictions stackable.
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X, y))
assert list(X.columns) == list(X_test.columns)
print(X.shape, X_test.shape)

C:\Users\89671\anaconda3\envs\kaggle-playground\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


W1010 03:34:53.136000 15116 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


device: cuda NVIDIA GeForce RTX 3070 Ti | torch 2.14.1+cu126 | pytabkit 1.7.3 | networks per model: 4


(699635, 92) (299844, 92)


## 1. Input frame (identical to `10`)
Rich columns without `orig_prob` and the `key_*` ids, plus 15 categorical twins (route, age, 13 ratings). The 7 `te_*` columns are added per fold by `add_te`. Category lists come from train+test together, so no fold meets an unknown level. See `10` for the reasoning.

In [2]:
RATINGS = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
           'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
           'checkin_service','cleanliness']
TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS}}
FULL = pd.concat([X, X_test], ignore_index=True)
TWIN_LEVELS = {t: sorted(FULL[s].astype(int).unique()) for t, s in TWINS.items()}
KEYS = [c for c in X.columns if c.startswith('key_')]      # integer ids for target encoding only, not fed as numbers
TE_SOURCES = [c for c in ['flight_distance', 'age', 'fd_div10', 'fd_mod1000'] + KEYS if c in X.columns]
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(TWINS)

def nn_frame(D):
    D = D.drop(columns=['orig_prob'] + KEYS).copy()
    for t, s in TWINS.items():
        D[t] = pd.Categorical(D[s].astype(int), categories=TWIN_LEVELS[t])
    assert not D.isna().any().any()
    return D

print('TE sources:', len(TE_SOURCES), TE_SOURCES, '| input columns before TE:', nn_frame(X.iloc[:5]).shape[1])

TE sources: 7 ['flight_distance', 'age', 'fd_div10', 'fd_mod1000', 'key_age_class', 'key_fd_class', 'key_age_travel'] | input columns before TE: 103


## 2. The two recipes
| Parameter | Reference (`10`) | v2 | What it does |
|---|---|---|---|
| `lr_sched` | `lin_cos_log_15` | `flat_anneal` | reference: rising cosine cycles of growing length; v2: constant learning rate for 60% of training, then a cosine decay to ~0 |
| `wd` | 0.0236 | 0.015 | weight decay (pull of the weights towards 0); v2 regularises less |
| `p_drop_sched` | `expm4t` | `invsqrtp1e-3` | dropout multiplier over time: reference `exp(−4t)` (×0.67 at t = 0.1); v2 `√(1e-3/(t+1e-3))` (×0.1 at t = 0.1), i.e. dropout is practically off after the first few percent |
| `n_epochs` | 4 | 3 | passes over the training part |
| `n_ens` | 4 | 4 | networks trained side by side (unchanged on purpose) |

Everything else (lr 0.053, PLR embeddings, 512-256-128, label smoothing, …) is shared. The cell **checks that every schedule name exists in the installed pytabkit**: an unknown name must stop the notebook, not be silently replaced.

In [3]:
PARAMS_REF = dict(
    n_epochs=1 if SMOKE else 4, n_ens=2 if SMOKE else N_ENS, batch_size=256, use_early_stopping=False,
    val_metric_name='1-auc_ovr', device=DEVICE, random_state=0, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False,
    bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
)
# v2 = reference with exactly four changes (n_ens stays the same)
PARAMS_V2 = {**PARAMS_REF, 'lr_sched': 'flat_anneal', 'wd': 0.015, 'p_drop_sched': 'invsqrtp1e-3', 'n_epochs': 1 if SMOKE else 3}
RECIPES = {'ref': PARAMS_REF, 'v2': PARAMS_V2}

# Guard: every schedule name must be a branch of pytabkit's schedule factory in THIS installed version.
SRC = inspect.getsource(pt_sched)
for rname, P in RECIPES.items():
    for k in ['lr_sched', 'wd_sched', 'p_drop_sched', 'ls_eps_sched']:
        assert f"sched_type == '{P[k]}'" in SRC, f'{rname}: {k}={P[k]!r} is not defined in pytabkit {PYTABKIT_VERSION}'
print('schedules found in pytabkit', PYTABKIT_VERSION, '| changed keys:', {k: (PARAMS_REF[k], PARAMS_V2[k]) for k in PARAMS_V2 if PARAMS_V2[k] != PARAMS_REF[k]})

# In-fold target encoding, identical to 10: fitted on the fold's TRAINING part (inner cv=5, so training rows are
# never encoded with their own label), only applied to validation and test.
def add_te(Xtr, ytr, others):
    te = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
    cols = [f'te_{c}' for c in TE_SOURCES]
    tr_te = pd.DataFrame(te.fit_transform(Xtr[TE_SOURCES].astype(int), ytr), columns=cols, index=Xtr.index).astype('float32')
    outs = [pd.DataFrame(te.transform(o[TE_SOURCES].astype(int)), columns=cols, index=o.index).astype('float32') for o in others]
    return [pd.concat([D, T], axis=1) for D, T in zip([Xtr] + list(others), [tr_te] + outs)]

def fit_predict(params, tr_idx, va_idx, seed=None, rows=None):
    seed = SEED if seed is None else seed
    tr_idx = tr_idx if rows is None else tr_idx[:rows]
    Xtr, Xva, Xte = add_te(X.iloc[tr_idx], y[tr_idx], [X.iloc[va_idx], X_test])
    Xtr, Xva, Xte = nn_frame(Xtr), nn_frame(Xva), nn_frame(Xte)
    model = RealMLP_TD_Classifier(**{**params, 'random_state': seed})
    # X_val is passed only for logging; early stopping is off, so validation labels never choose anything.
    model.fit(Xtr, y[tr_idx], X_val=Xva, y_val=y[va_idx], cat_col_names=CAT_COLS)
    p_va, p_te = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return p_va, p_te

schedules found in pytabkit 1.7.3 | changed keys: {'n_epochs': (4, 3), 'wd': (0.0236, 0.015), 'lr_sched': ('lin_cos_log_15', 'flat_anneal'), 'p_drop_sched': ('expm4t', 'invsqrtp1e-3')}


## 3. Timing probe
v2 for one epoch on 50,000 rows, extrapolated to a full fold (≈ 560,000 rows × 3 epochs). An upper bound: setup and the prediction of 320k rows are included.

In [4]:
tr0, va0 = FOLDS[0]
t0 = time.time()
_ = fit_predict({**PARAMS_V2, 'n_epochs': 1}, tr0, va0[:20000], rows=50000)
t_probe = time.time() - t0
est = t_probe * (len(tr0) / 50000) * PARAMS_V2['n_epochs']
print(f'probe: {t_probe:.0f} s for 50k rows x 1 epoch -> rough upper bound for one fold of v2: {est/60:.1f} min')

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


probe: 8 s for 50k rows x 1 epoch -> rough upper bound for one fold of v2: 4.4 min


## 4. Fold 0 — paired benchmark
Both recipes, seed 0, this machine. Each result is checkpointed (`preds/realmlp_rich_refgpu_folds/`, `preds/realmlp_rich_v2_folds/`), so rerunning the cell loads instead of retraining. The Mac seeds of `10` are read from their checkpoints for reference.

In [5]:
def ck_dir(tag, seed):
    return PREDS_DIR / (f'realmlp_rich_{tag}_folds' if seed == 0 else f'realmlp_rich_{tag}_folds_s{seed}')

def run_fold(tag, params, k, seed=None):
    seed = SEED if seed is None else seed
    d = ck_dir(tag, seed); fo, ft = d / f'oof_{k}.npy', d / f'test_{k}.npy'
    if fo.exists() and ft.exists():
        return np.load(fo), np.load(ft), None
    t0 = time.time(); p_va, p_te = fit_predict(params, *FOLDS[k], seed=seed, rows=60000 if SMOKE else None)
    if not SMOKE:
        d.mkdir(exist_ok=True); np.save(fo, p_va); np.save(ft, p_te)
    return p_va, p_te, (time.time() - t0) / 60

va0 = FOLDS[0][1]
rows = []
for s, d in [(0, 'realmlp_rich_folds'), (1, 'realmlp_rich_folds_s1'), (2, 'realmlp_rich_folds_s2')]:
    f = PREDS_DIR / d / 'oof_0.npy'
    if f.exists(): rows.append({'run': f'ref, Mac CPU, seed {s} (from 10)', 'fold 0 AUC': roc_auc_score(y[va0], np.load(f)), 'min': None})
P0 = {}
for tag in (['ref', 'v2'] if RUN_REF else ['v2']):
    p_va, p_te, mins = run_fold('refgpu' if tag == 'ref' else 'v2', RECIPES[tag], 0)
    P0[tag] = (p_va, p_te)
    rows.append({'run': f'{tag}, {DEVICE}, seed {SEED}', 'fold 0 AUC': roc_auc_score(y[va0], p_va), 'min': mins})
    print(rows[-1], flush=True)
bench = pd.DataFrame(rows); print(bench.round(5).to_string(index=False))

best_mac = bench.loc[bench.run.str.contains('Mac'), 'fold 0 AUC'].max()
a_v2 = roc_auc_score(y[va0], P0['v2'][0])
print(f'\nv2 - best Mac seed : {a_v2 - best_mac:+.5f}')
if 'ref' in P0:
    a_ref = roc_auc_score(y[va0], P0['ref'][0])
    print(f'v2 - ref (same machine, same seed): {a_v2 - a_ref:+.5f}   | ref GPU - Mac seed 0: {a_ref - bench["fold 0 AUC"].iloc[0]:+.5f}')
print('acceptance for the next stage: gain >= +0.0002 on fold 0')

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


`Trainer.fit` stopped: `max_epochs=4` reached.


...

TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


{'run': 'v2, cuda, seed 0', 'fold 0 AUC': 0.9613780150723411, 'min': 1.1660586595535278}


                           run  fold 0 AUC     min
ref, Mac CPU, seed 0 (from 10)     0.96134     NaN
ref, Mac CPU, seed 1 (from 10)     0.96141     NaN
ref, Mac CPU, seed 2 (from 10)     0.96133     NaN
             ref, cuda, seed 0     0.96138 1.56855
              v2, cuda, seed 0     0.96138 1.16606

v2 - best Mac seed : -0.00003
v2 - ref (same machine, same seed): +0.00000   | ref GPU - Mac seed 0: +0.00004
acceptance for the next stage: gain >= +0.0002 on fold 0


## 5. All 5 folds (only if fold 0 passed)
Resumable: finished folds are loaded from `preds/realmlp_rich_v2_folds[_s<seed>]/`. Fold 0 is reused from section 4. Off by default.

In [6]:
RUN_ALL = False
if RUN_ALL and not SMOKE:
    oof = np.full(len(X), np.nan); tests = []
    for k, (tr, va) in enumerate(FOLDS):
        p_va, p_te, mins = run_fold('v2', PARAMS_V2, k)
        oof[va] = p_va; tests.append(p_te)
        print(f'fold {k}: AUC {roc_auc_score(y[va], p_va):.5f}' + ('' if mins is None else f'  ({mins:.1f} min)'), flush=True)
    print(f'\nOOF AUC (RealMLP-rich v2): {roc_auc_score(y, oof):.5f}')
    sfx = '' if SEED == 0 else f'_s{SEED}'
    np.save(PREDS_DIR / f'oof_realmlp_rich_v2{sfx}.npy', oof); np.save(PREDS_DIR / f'test_realmlp_rich_v2{sfx}.npy', np.mean(tests, axis=0))

## 6. Average the seeds (only if section 5 ran for several seeds)
Logit average of `oof_realmlp_rich_v2[_s<seed>].npy` → `oof_realmlp_rich_v2_avg.npy` / `test_…`, same convention as `10`.

In [7]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
sig = lambda z: 1 / (1 + np.exp(-z))
seeds = [(0, PREDS_DIR / 'oof_realmlp_rich_v2.npy', PREDS_DIR / 'test_realmlp_rich_v2.npy')] + \
        [(s, PREDS_DIR / f'oof_realmlp_rich_v2_s{s}.npy', PREDS_DIR / f'test_realmlp_rich_v2_s{s}.npy') for s in range(1, 10)]
seeds = [t for t in seeds if t[1].exists() and t[2].exists()]
if len(seeds) > 1:
    O_ = [lg(np.load(o)) for _, o, _ in seeds]; T_ = [lg(np.load(t)) for _, _, t in seeds]
    for (s, _, _), o in zip(seeds, O_): print(f'seed {s}: OOF AUC {roc_auc_score(y, o):.5f}')
    print(f'average of {len(seeds)} seeds: OOF AUC {roc_auc_score(y, np.mean(O_, axis=0)):.5f}')
    np.save(PREDS_DIR / 'oof_realmlp_rich_v2_avg.npy', sig(np.mean(O_, axis=0))); np.save(PREDS_DIR / 'test_realmlp_rich_v2_avg.npy', sig(np.mean(T_, axis=0)))
else:
    print('seeds found:', len(seeds), '-> nothing to average')

seeds found: 0 -> nothing to average


## Results & decisions

**Fold 0, 4 networks, seed 0** (RTX 3070 Ti, torch 2.14.1+cu126, pytabkit 1.7.3)

| Run | Fold 0 AUC | Time |
|---|---|---|
| ref, Mac CPU, seeds 0 / 1 / 2 (from `10`) | 0.96134 / 0.96141 / 0.96133 | ≈ 4.8 min |
| ref, GPU, seed 0 | 0.96138 | 1.6 min |
| **v2, GPU, seed 0** | **0.96138** | 1.2 min |

| Comparison | Difference | Verdict |
|---|---|---|
| v2 − ref (same machine, same seed) | **+0.00000** | no effect |
| v2 − best Mac seed | −0.00003 | below the +0.0002 bar |
| ref GPU − ref Mac seed 0 | +0.00004 | GPU reproduces the Mac within seed noise (spread of 3 seeds: 0.00007) |

**Decision: v2 is rejected as a replacement recipe; step closed.** The recipe changes the ranking of rows (see below) but not its quality: RealMLP-rich is not sensitive to these schedule / regularisation settings, so a hyper-parameter search of RealMLP (backlog item 12) is not triggered.

**Side observation (not a decision):** v2 makes *different* errors from the reference — Spearman ref–v2 0.9932 vs 0.9955–0.9958 between seeds of the same recipe. Averaging ref + v2 gains +0.00008 over the better member, a seed pair only +0.00002–0.00003. Added to the 3-seed average of `10`, v2 gives +0.00005 on fold 0 — noise level for one fold. Useful later: if more RealMLP seeds are trained (plan step 7), mixing recipes is likely a better use of compute than repeating one recipe.

**Practical:** on the home GPU a RealMLP-rich fold takes ≈ 1.5 min (4 networks), ≈ 3× faster than the Mac CPU.


## Glossary
* **Recipe** — training settings (schedules, regularisation, number of epochs), as opposed to input features.
* **Schedule** — a multiplier that changes a value (learning rate, weight decay, dropout) as training time `t` goes from 0 to 1.
* **`flat_anneal`** — constant learning rate for the first 60% of training, then cosine decay to 1e-5.
* **`invsqrtp1e-3`** — dropout multiplier √(1e-3 / (t + 1e-3)): 1 at the start, 0.1 at t = 0.1, 0.03 at the end.
* **Paired comparison** — two variants trained under identical conditions (data, fold, seed, machine).
* **In-fold target encoding** — category → mean target, fitted only on the training part of the fold (see `10`).